# Treinamento de modelos — Previsão de falência de pequenas empresas

Este notebook complementa `AnlisedeDados.ipynb`, que fazia a análise exploratória dos cinco arquivos. Nesta etapa, os dados são preparados e dois classificadores são comparados: Random Forest e XGBoost.

**Alvo:** `class` — 0 = não falida; 1 = falida.  
**Cuidados:** dados desbalanceados, imputação aprendida apenas no treino, seleção de atributos sem usar validação/teste e avaliação no teste independente.  
**Arquivos esperados:** `dados/1year.csv` até `dados/5year.csv`.  
**Para interpretar os resultados completos:** consulte `RELATORIO_TREINAMENTO.md`.

## 1. Resultados obtidos nesta execução

Após concatenar os cinco arquivos e remover 401 duplicatas exatas, foram usados **43.004 registros**, dos quais **2.083 (4,84%)** pertencem à classe falida. O conjunto foi separado em treino (60%), validação (20%) e teste (20%).

No teste, o Random Forest alcançou precisão de 0,188, recall de 0,504, F1 de 0,274 e ROC-AUC de 0,823 para a classe falida. O XGBoost alcançou precisão de 0,181, recall de 0,484, F1 de 0,264 e ROC-AUC de 0,801. O Random Forest foi selecionado pelo melhor F1 de validação.

A precisão baixa mostra que ainda ocorrem falsos positivos; portanto, as previsões devem priorizar análises adicionais, e não decisões automatizadas.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from treinar_modelos import main

PASTA_DADOS = Path("dados")
PASTA_SAIDA = Path("artifacts")

# Execute esta célula para reproduzir o treinamento e gerar os arquivos de saída.
resultado = main(PASTA_DADOS, PASTA_SAIDA)

In [ ]:
# 2. Comparação das métricas no teste
metricas = pd.read_csv(PASTA_SAIDA / "comparacao_modelos.csv")
display(metricas)

In [ ]:
# 3. Atributos usados pelo modelo principal
atributos = pd.read_csv(PASTA_SAIDA / "atributos_selecionados.csv")
display(atributos)

In [ ]:
# 4. Resumo dos resultados e do modelo selecionado
detalhes = json.loads((PASTA_SAIDA / "resultados_treinamento.json").read_text(encoding="utf-8"))
print("Modelo escolhido:", detalhes["melhor_modelo_selecionado"])
print("Limiar de decisão:", detalhes["limiar_final"])
print("Divisão:", detalhes["divisao"])
print("Matriz de confusão no teste:")
print(detalhes["matriz_confusao_teste_modelo_escolhido"])

## Limitações e próximos passos

- A classe falida representa uma pequena parcela dos registros, então acurácia sozinha é inadequada.
- O limiar foi ajustado para F1 na validação; outro limiar pode ser apropriado se o custo de falso negativo e falso positivo mudar.
- O score não foi calibrado como probabilidade real.
- Antes de produção, valide em dados externos/mais recentes, revise o cálculo de cada indicador financeiro e avalie calibração e estabilidade do modelo.